# Jev Latest · OpenRouter Decisions API 示例

> 页面：[TypeSafe: Jev Latest](https://openrouter.ai/~typesafe/jev-latest)（`~typesafe/jev-latest` → 现网 Jev 1.13）
> 配套：[TypeSafe全景与System-One.md](./TypeSafe全景与System-One.md) · [Jev接入与工程实践.md](./Jev接入与工程实践.md)
> 本 notebook **照抄页面 Quick Start**：`POST /api/alpha/decisions`，一份 `state` + 三条 typed questions，代码读 `answers` 再分支。
> Jev **不生成文本**。不要把它丢进 `/api/v1/chat/completions`。页面 FAQ 写「OpenAI 兼容」对这个模型不成立。

**怎么跑**：逐格 `Shift+Enter`。§1–§2 只构造、不出网。§3 有 `OPENROUTER_API_KEY` 才真打；没有 key 用页面同构的 fixture，后面格子照样能跑。

| 节 | 对照页面 | 做什么 |
|---|---|---|
| §1 | Quick Start 说明 | 端点 / 模型 slug / 三种问题 |
| §2 | Python 请求体 | 原文 payload（payouts + noul/choice/score） |
| §3 | `requests.post` | 发出去；无 key 则跳过 |
| §4 | `answers[...]` + `if noul > 0.8` | 按页面读字段、升级账单 |
| §5 | 「Your code owns the workflow」 | 同一份 answers 上扩一点门闩 |


## 1. 页面在说什么

Jev 通过 **Decisions API** 回答关于 `state` 的类型化问题，返回校准概率：

- **noul**：是/否概率（0=否，1=是）
- **choice**：从你给的选项里挑一个，带全分布
- **score**：落在你写的有序量表上

`state` 可以是字符串、对象或数组。工作流归你的代码，所以适合路由 / 排序 / 校验，而不是聊天。


In [ ]:
ENDPOINT = "https://openrouter.ai/api/alpha/decisions"
MODEL = "~typesafe/jev-latest"

print("端点 :", ENDPOINT)
print("模型 :", MODEL, "  （~ 前缀 = 永远指向 Jev 家族最新）")
print("钉版本用 typesafe/jev-1.13（没有 ~）")
print("价格 : $0.042 / M 输入，输出 $0")
print("窗口 : OpenRouter 公示 32,000 token")
print()
print("不是 Chat Completions。")


## 2. 原文请求体

下面就是页面示例：工单原文当 `state`，一次问紧急度、部门、愤怒程度。

noul 的 `true`/`false`、choice 的选项，页面写在问题对象**顶层**（官方原生文档则放进 `criteria`）。本 notebook 跟页面走。


In [ ]:
import json

STATE = "Help! My payouts have been failing for 3 days."

PAYLOAD = {
    "model": MODEL,
    "state": STATE,
    "questions": {
        "is_urgent": {
            "type": "noul",
            "instructions": "Does this message convey urgency?",
            "true": "Explicitly time-sensitive",
            "false": "No urgency expressed",
        },
        "department": {
            "type": "choice",
            "instructions": "Which team should handle this?",
            "billing": "Payments, invoicing, refunds",
            "technical": "Bugs, outages, integrations",
            "sales": "Pricing, upgrades, new accounts",
        },
        "frustration": {
            "type": "score",
            "instructions": "How frustrated is the customer?",
            "criteria": ["Calm", "Frustrated", "Very angry"],
        },
    },
}

print(json.dumps(PAYLOAD, indent=2, ensure_ascii=False))


## 3. 发出 `POST /api/alpha/decisions`

页面用 `requests.post`。这里用标准库 `urllib` 做同一件事（零额外依赖）；有 `requests` 也可以换成页面原文。

请求头里的 `HTTP-Referer` / `X-OpenRouter-Title` 可选，给 openrouter.ai 榜单归因用。密钥只读环境变量 `OPENROUTER_API_KEY`。


In [ ]:
import os
import urllib.error
import urllib.request

# 页面量级 fixture，无 key 时后面格子继续用
FIXTURE = {
    "model": "jev-1.13.0",
    "answers": {
        "is_urgent": {"type": "noul", "noul": 0.95},
        "department": {
            "type": "choice",
            "choice": "billing",
            "confidence": 0.80,
            "probabilities": {"billing": 0.87, "technical": 0.13, "sales": 0.0},
        },
        "frustration": {
            "type": "score",
            "score": 1.04,
            "confidence": 0.94,
            "legend": {"0": "Calm", "1": "Frustrated", "2": "Very angry"},
            "probabilities": {"0": 0.0, "1": 0.96, "2": 0.04},
        },
    },
    "usage": {"input_tokens": 426, "output_tokens": 73},
}

HEADERS = {
    "Authorization": f"Bearer {os.environ.get('OPENROUTER_API_KEY', '')}",
    "Content-Type": "application/json",
    "HTTP-Referer": "https://openrouter.ai/~typesafe/jev-latest",
    "X-OpenRouter-Title": "jev-notebook",
}


def post_decisions(payload):
    req = urllib.request.Request(
        ENDPOINT,
        data=json.dumps(payload).encode(),
        headers=HEADERS,
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read().decode())


LIVE = False
if not os.environ.get("OPENROUTER_API_KEY"):
    print("未设置 OPENROUTER_API_KEY，用页面同构 fixture（不出网）。")
    print("真打：export OPENROUTER_API_KEY=... 然后重跑本格。")
    response_json = FIXTURE
else:
    try:
        response_json = post_decisions(PAYLOAD)
        LIVE = True
        print("渠道: OpenRouter Decisions API")
        print("POST ", ENDPOINT)
    except urllib.error.HTTPError as e:
        print(f"HTTP {e.code}: {e.read().decode()[:500]}")
        print("回退 fixture。")
        response_json = FIXTURE
    except Exception as e:
        print(f"{type(e).__name__}: {e}")
        print("回退 fixture。")
        response_json = FIXTURE

print("响应 model =", response_json.get("model"))
print("usage      =", response_json.get("usage"))
print("live       =", LIVE)


## 4. 按页面读 `answers`，代码做分支

页面原文：

```python
answers = response.json()["answers"]
print(answers["is_urgent"]["noul"])
print(answers["department"]["choice"], answers["department"]["probabilities"])
print(answers["frustration"]["score"])

if answers["is_urgent"]["noul"] > 0.8 and answers["department"]["choice"] == "billing":
    pass  # escalate_to_billing(...)
```

noul 是 0（否）到 1（是）的概率；choice / score 带全分布。阈值 `0.8` 是页面示例，不是官方魔法数。


In [ ]:
answers = response_json["answers"]

# noul is a probability from 0 (no) to 1 (yes); choice and score carry the full distribution.
print(answers["is_urgent"]["noul"])
print(answers["department"]["choice"], answers["department"]["probabilities"])
print(answers["frustration"]["score"])

if answers["is_urgent"]["noul"] > 0.8 and answers["department"]["choice"] == "billing":
    print("→ escalate_to_billing(...)")
else:
    print("→ 不升级。urgency=", answers["is_urgent"]["noul"], "team=", answers["department"]["choice"])


## 5. 同一份 answers：工作流仍归代码

页面说 *Your code owns the workflow*。部门 choice 还有 `confidence`；noul 没有单独的 confidence（靠近 0.5 自己当不确定）。`score` 是档位编号的加权均值，不要只看一个数。


In [ ]:
dept = answers["department"]
frust = answers["frustration"]
urgent = answers["is_urgent"]["noul"]

print("department.confidence =", dept.get("confidence"))
print("frustration.legend    =", frust.get("legend"))
print("frustration.probs     =", frust.get("probabilities"))

# 高风险动作把门槛抬高：页面用 0.8 做紧急度；账单升级再看部门置信
if urgent > 0.8 and dept["choice"] == "billing":
    if dept.get("confidence", 1.0) >= 0.9:
        action = "auto_escalate"
    else:
        action = "confirm_then_escalate"
else:
    action = "normal_queue"

print("action =", action)
print()
print("改阈值只改本格数字，不必改 questions。")


## 6. 页面 `requests` 原文（对照用）

本机若已 `pip install requests`，下面就是 OpenRouter 文档那一截。没装就跳过——§3 的 urllib 已经做完同一件事。


In [ ]:
try:
    import requests
except ImportError:
    print("未安装 requests。§3 已用 urllib 打过同一端点。")
    print("要对齐页面原文：pip install requests 后重跑本格。")
else:
    key = os.environ.get("OPENROUTER_API_KEY")
    if not key:
        print("有 requests，但没 OPENROUTER_API_KEY。不发请求。")
    else:
        response = requests.post(
            url=ENDPOINT,
            headers={
                "Authorization": f"Bearer {key}",
                "Content-Type": "application/json",
                "HTTP-Referer": "https://openrouter.ai/~typesafe/jev-latest",
                "X-OpenRouter-Title": "jev-notebook",
            },
            data=json.dumps(PAYLOAD),
        )
        response.raise_for_status()
        answers = response.json()["answers"]
        print(answers["is_urgent"]["noul"])
        print(answers["department"]["choice"], answers["department"]["probabilities"])
        print(answers["frustration"]["score"])
        if answers["is_urgent"]["noul"] > 0.8 and answers["department"]["choice"] == "billing":
            print("→ escalate_to_billing(...)")
